In [3]:
cd /content/drive/MyDrive/Abbreviations/

/content/drive/MyDrive/Abbreviations


In [7]:
import pandas as pd
import numpy as np
import sys

# --- 1. Define File Names ---
file_db = 'DatabaseRef004New.csv'
file_abbr = 'Abbreviations.csv'

# --- 2. Load Data ---
df_db = None
df_abbr = None

try:
    # Load the main database file
    df_db = pd.read_csv(file_db)
    print(f"Successfully loaded '{file_db}'.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure '{file_db}' is uploaded to your Colab environment.")
    sys.exit() # Exit if main file is not found

try:
    # Load the abbreviations lookup table
    df_abbr = pd.read_csv(file_abbr)
    print(f"Successfully loaded '{file_abbr}'.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure '{file_abbr}' is uploaded to your Colab environment.")
    # Proceed, but replacement/location steps will have no effect
    df_abbr = pd.DataFrame({'本书简称': [], '出版单位全称': [], '所在地': []})


# --- 3. Data Cleaning and Preparation ---

# Clean lookup table columns
df_abbr['本书简称'] = df_abbr['本书简称'].fillna('').astype(str).str.strip()
df_abbr['出版单位全称'] = df_abbr['出版单位全称'].fillna('').astype(str).str.strip()
df_abbr['所在地'] = df_abbr['所在地'].fillna('').astype(str).str.strip()

# Create a mapping dictionary for abbreviation replacement
# Key: '本书简称' (Abbreviation), Value: '出版单位全称' (Full Name)
abbr_to_full = dict(zip(df_abbr['本书简称'], df_abbr['出版单位全称']))

# Filter the mapping to exclude cases where Abbreviation == Full Name or Abbreviation is empty
abbr_to_full_filtered = {
    abbr: full for abbr, full in abbr_to_full.items() if abbr and abbr != full
}

# --- 4. Replace Abbreviations with Full Names ---

if abbr_to_full_filtered:
    # Use .replace() to substitute all matching abbreviations with their full names
    df_db['出版社'] = df_db['出版社'].replace(abbr_to_full_filtered)
    print("\n✅ Abbreviations in '出版社' column have been replaced with full names.")
else:
    print("\n⚠️ No valid abbreviation-to-full-name mappings found or needed.")


# --- 5. Add '所在地' (Location) Column via Merge ---

# Prepare the minimal lookup table for location merging.
# We use '出版单位全称' as the key because all abbreviations were just converted to full names.
df_location_map = df_abbr[['出版单位全称', '所在地']].drop_duplicates()

# Perform a **left merge** to add the '所在地' column.
# This attaches the '所在地' based on matching df_db['出版社'] with df_location_map['出版单位全称'].
df_db = df_db.merge(
    df_location_map,
    left_on='出版社',
    right_on='出版单位全称',
    how='left'
)

# Clean up: drop the redundant '出版单位全称' column added during the merge
if '出版单位全称' in df_db.columns:
    df_db = df_db.drop(columns='出版单位全称')

print("✅ New '所在地' column has been added with corresponding location names.")

# --- 6. Display and Save Results ---

# Display the first few rows to show the changes
print("\n--- First 10 rows of the Updated DataFrame (showing '出版社' and '所在地') ---")
print(df_db[['出版社', '所在地']].head(10))

# Save the updated DataFrame to a new CSV file
output_file = 'DatabaseRef004_Processed.csv'
df_db.to_csv(output_file, index=False, encoding='utf-8')
print(f"\n✨ Updated data saved to '{output_file}'.")

print("\nScript finished.")

Successfully loaded 'DatabaseRef004New.csv'.
Successfully loaded 'Abbreviations.csv'.

✅ Abbreviations in '出版社' column have been replaced with full names.
✅ New '所在地' column has been added with corresponding location names.

--- First 10 rows of the Updated DataFrame (showing '出版社' and '所在地') ---
       出版社  所在地
0       广益  NaN
1  大众美术出版社   上海
2      NaN  NaN
3      NaN  NaN
4  大众美术出版社   上海
5   晨光出版公司   上海
6    教育出版社   上海
7     华东书店   上海
8  大众美术出版社   上海
9     中南新华  NaN

✨ Updated data saved to 'DatabaseRef004_Processed.csv'.

Script finished.


In [11]:
import pandas as pd
import numpy as np

# Define file names
processed_file = 'DatabaseRef004_Processed.csv'
abbr_file = 'Abbreviations.csv'

# Load the processed database file
try:
    df_processed = pd.read_csv(processed_file)
    print(f"Successfully loaded '{processed_file}'.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure '{processed_file}' is in your Colab environment.")
    sys.exit()

# Load the abbreviations lookup table
try:
    df_abbr = pd.read_csv(abbr_file)
    print(f"Successfully loaded '{abbr_abbr}'.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure '{abbr_file}' is in your Colab environment.")
    # If abbreviations file is missing, we can't fill missing locations
    print("Cannot check and fill missing locations without the Abbreviations file.")
    sys.exit()

# Clean and prepare the abbreviations data for merging
df_abbr['出版单位全称'] = df_abbr['出版单位全称'].fillna('').astype(str).str.strip()
df_abbr['所在地'] = df_abbr['所在地'].fillna('').astype(str).str.strip()

# Create a location mapping from the abbreviations file
# Ensure only valid location entries are used for mapping
location_map = df_abbr[df_abbr['所在地'] != ''].set_index('出版单位全称')['所在地'].to_dict()

print("\nChecking for and filling missing locations...")

# List to store information about filled locations
filled_locations_list = []

# Iterate through rows in the processed dataframe
for index, row in df_processed.iterrows():
    publisher = row['出版社']
    location = row['所在地']

    # Check if location is missing (empty string or NaN after previous steps)
    # and if the publisher exists in our location map
    if (pd.isna(location) or location == '') and publisher in location_map:
        # Get the location from the map
        correct_location = location_map[publisher]

        # Store the original and filled location information
        filled_locations_list.append({
            'Index': index,
            'Publisher': publisher,
            'Original Location': location, # This will be NaN or ''
            'Filled Location': correct_location
        })

        # Fill the missing location in the processed dataframe
        df_processed.at[index, '所在地'] = correct_location


# Count how many missing locations were filled
filled_count = len(filled_locations_list)

print(f"\n✅ Finished checking and filling missing locations. {filled_count} entries were updated.")

# Display the list of filled locations
if filled_locations_list:
    print("\n--- List of Filled Locations ---")
    for entry in filled_locations_list:
        print(f"Index {entry['Index']}: Publisher '{entry['Publisher']}' - Filled '{entry['Filled Location']}' (Original was empty/NaN)")
else:
    print("\nNo missing locations were filled based on the Abbreviations file.")


# Re-save the updated DataFrame
output_file_updated = 'DatabaseRef004_Processed_UpdatedLocations.csv'
df_processed.to_csv(output_file_updated, index=False, encoding='utf-8')

print(f"\n✨ Updated data with filled locations saved to '{output_file_updated}'.")

Successfully loaded 'DatabaseRef004_Processed.csv'.
Successfully loaded 'Abbreviations.csv'.

Checking for and filling missing locations...

✅ Finished checking and filling missing locations. 0 entries were updated.

No missing locations were filled based on the Abbreviations file.

✨ Updated data with filled locations saved to 'DatabaseRef004_Processed_UpdatedLocations.csv'.


In [12]:
import pandas as pd
import numpy as np

# Define file names
processed_file = 'DatabaseRef004_Processed.csv'
output_file_missing_locations = 'DatabaseRef004_MissingLocations.csv'

# Load the processed database file
try:
    df_processed = pd.read_csv(processed_file)
    print(f"Successfully loaded '{processed_file}'.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure '{processed_file}' is in your Colab environment.")
    sys.exit()

# Filter rows where '所在地' is missing (NaN or empty string)
# Replace empty strings with NaN first to treat both as missing
df_processed['所在地'] = df_processed['所在地'].replace('', np.nan)
df_missing_locations = df_processed[df_processed['所在地'].isna()]

print(f"\nFound {len(df_missing_locations)} rows with missing location data.")

# Save the filtered DataFrame to a new CSV file
df_missing_locations.to_csv(output_file_missing_locations, index=False, encoding='utf-8')

print(f"✨ Rows with missing locations saved to '{output_file_missing_locations}'.")

# Optional: Display the first few rows of the new file
# print("\n--- First 10 rows of the file with missing locations ---")
# print(df_missing_locations.head(10))

Successfully loaded 'DatabaseRef004_Processed.csv'.

Found 2434 rows with missing location data.
✨ Rows with missing locations saved to 'DatabaseRef004_MissingLocations.csv'.


In [14]:
import pandas as pd

# Define file names
missing_locations_file = 'DatabaseRef004_MissingLocations.csv'
non_matching_publishers_file = 'DatabaseRef004_NonMatchingPublishers.csv'

# Load the CSV files
try:
    df_missing_locations = pd.read_csv(missing_locations_file)
    print(f"Successfully loaded '{missing_locations_file}'.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure '{missing_locations_file}' is in your Colab environment.")
    sys.exit()

try:
    df_non_matching_publishers = pd.read_csv(non_matching_publishers_file)
    print(f"Successfully loaded '{non_matching_publishers_file}'.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure '{non_matching_publishers_file}' is in your Colab environment.")
    sys.exit()

print("\nComparing the two dataframes...")

# Check if the two dataframes are identical
# This checks both the data and the column order
are_identical = df_missing_locations.equals(df_non_matching_publishers)

if are_identical:
    print("✅ The data in 'DatabaseRef004_MissingLocations.csv' and 'DatabaseRef004_NonMatchingPublishers.csv' is identical.")
else:
    print("⚠️ The data in 'DatabaseRef004_MissingLocations.csv' and 'DatabaseRef004_NonMatchingPublishers.csv' is NOT identical.")

# You can also check if they have the same shape (number of rows and columns)
if df_missing_locations.shape == df_non_matching_publishers.shape:
    print(f"They have the same shape: {df_missing_locations.shape}")
else:
    print(f"They have different shapes: {df_missing_locations.shape} vs {df_non_matching_publishers.shape}")

# You could also check if the indices are the same
# are_indices_identical = df_missing_locations.index.equals(df_non_matching_publishers.index)
# print(f"Indices are identical: {are_indices_identical}")

# And check if the columns are the same (regardless of order)
# are_columns_identical = set(df_missing_locations.columns) == set(df_non_matching_publishers.columns)
# print(f"Columns are identical (ignoring order): {are_columns_identical}")

Successfully loaded 'DatabaseRef004_MissingLocations.csv'.
Successfully loaded 'DatabaseRef004_NonMatchingPublishers.csv'.

Comparing the two dataframes...
✅ The data in 'DatabaseRef004_MissingLocations.csv' and 'DatabaseRef004_NonMatchingPublishers.csv' is identical.
They have the same shape: (2434, 11)
